In [71]:
import os
import torch
from torch.utils.data import DataLoader

from dataset import (
    MaridaDatasetLoader,
    find_patch_bases,
    do_img_conf_mask_exist,
)
from preprocessing import (
    normalize_image,
    set_low_conf_for_nan,
    apply_augmentations,
    build_conf_ignore_mask,
    apply_ignore_index_to_target,
    flatten_for_rf,
    compute_dataset_stats,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [72]:

########################
# 1. TRANSFORMS
########################

def train_transform(img, mask, conf):
    """
    Préprocessing appliqué pendant l'entraînement :
    - normalisation
    - NaN/Inf -> conf=3 + img nettoyée
    - augmentations géométriques
    """
    # Normalisation (simple float)
    img = normalize_image(img)

    # Gérer les NaN/Inf -> conf = 3, img nettoyée
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)

    # Augmentations
    img, mask, conf = apply_augmentations(
        img,
        mask,
        conf,
        p_hflip=0.5,
        p_vflip=0.5,
        p_rotate90=0.5,
    )

    return img, mask, conf


def val_transform(img, mask, conf):
    """
    Préprocessing pour validation / test :
    - normalisation
    - NaN/Inf -> conf=3
    PAS d'augmentations.
    """
    img = normalize_image(img)
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)
    return img, mask, conf


In [73]:
def fix_base_name(base):
    """
    Prend '1-12-19_48MYU_0'
    et retourne 'S2_1-12-19_48MYU/S2_1-12-19_48MYU_0'
    """
    # Exemple : base = "1-12-19_48MYU_0"
    tile = base.rsplit("_", 1)[0]      # → "1-12-19_48MYU"
    folder = "S2_" + tile              # → "S2_1-12-19_48MYU"
    full = f"{folder}/S2_{base}"       # → "S2_1-12-19_48MYU/S2_1-12-19_48MYU_0"
    return full

In [74]:
def train_transform(img, mask, conf):
    # 1) Normaliser (juste cast en float)
    img = normalize_image(img)

    # 2) Si NaN/Inf -> conf = 3 et remplacer NaN par 0
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)

    # 3) Augmentations (flip / rotate) seulement sur le train
    img, mask, conf = apply_augmentations(img, mask, conf)

    return img, mask, conf


def val_transform(img, mask, conf):
    # Val/test : on veut du déterministe → pas d’augmentations
    img = normalize_image(img)
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)
    return img, mask, conf


In [75]:
########################
# 2. CONSTRUCTION DES DATASETS
########################

def build_bases(folder):
    """Retourne une liste de bases pour lesquelles img + mask + conf existent."""
    all_bases = find_patch_bases(folder)
    bases = [b for b in all_bases if do_img_conf_mask_exist(folder, b)]
    print(f"{folder} : {len(bases)} patches valides trouvés.")
    return bases

def load_split_list(split_file):
    """
    Lit un fichier de split (train / val / test)
    et retourne une liste de bases (strings) relatives à 'patches/'.
    Exemple de ligne dans le fichier : S2A_.../patch_0001
    """
    bases = []
    with open(split_file, "r") as f:
        for line in f:
            name = line.strip()
            if not name:
                continue
            # au cas où quelqu'un aurait mis .tif dans le fichier
            if name.endswith(".tif"):
                name = name[:-4]
            bases.append(name)
    return bases

def make_dataloaders(data_root, batch_size=4):
    """
    data_root = dossier 'raw' qui contient :
        - patches/
        - splits/ (train, val, test)
    """
    patches_root = os.path.join(data_root, "patches")
    splits_dir   = os.path.join(data_root, "splits")

    train_file = os.path.join(splits_dir, "train_X.txt")
    val_file   = os.path.join(splits_dir, "val_X.txt")

    # Charger les listes de bases depuis les fichiers
    train_bases = load_split_list(train_file)
    val_bases   = load_split_list(val_file)
    train_bases = [fix_base_name(b) for b in train_bases]
    val_bases   = [fix_base_name(b) for b in val_bases]

    # Vérifier que les fichiers .tif / _cl / _conf existent
    train_bases = [
        b for b in train_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]
    val_bases = [
        b for b in val_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]

    print(f"Train : {len(train_bases)} patches valides.")
    print(f"Val   : {len(val_bases)} patches valides.")

    train_dataset = MaridaDatasetLoader(
    folder=patches_root,
    bases=train_bases,
    transform=train_transform,
    )

    val_dataset = MaridaDatasetLoader(
    folder=patches_root,
    bases=val_bases,
    transform=val_transform,
    )


    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, val_loader

In [76]:
def make_test_loader(data_root, batch_size=4):
    patches_root = os.path.join(data_root, "patches")
    splits_dir   = os.path.join(data_root, "splits")

    test_file = os.path.join(splits_dir, "test_X.txt")
    test_bases = load_split_list(test_file)
    test_bases = [
        b for b in test_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]

    test_dataset = MaridaDatasetLoader(
        folder=patches_root,
        bases=test_bases,
        transform=val_transform,   # pas d'augmentations pour test
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    return test_loader


In [77]:
########################
# 3. EXEMPLE: CALCUL DES STATS
########################

def compute_stats_on_whole_dataset(data_root, batch_size=4):
    """
    Exemple de calcul de mean/std globales sur le dataset (sans augmentation).
    On utilise val_transform (sans aug) ou une transform spéciale si tu préfères.
    """
    train_folder = os.path.join(data_root, "train_X.txt")
    val_folder   = os.path.join(data_root, "val_X.txt")

    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    # Dataset sans augmentation (val_transform)
    full_dataset = torch.utils.data.ConcatDataset([
        MaridaDatasetLoader(folder=train_folder, bases=train_bases, transform=val_transform),
        MaridaDatasetLoader(folder=val_folder, bases=val_bases, transform=val_transform),
    ])

    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)

    mean, std = compute_dataset_stats(full_loader)
    print("Mean per band:", mean)
    print("Std per band:", std)
    return mean, std

In [78]:
########################
# 4. EXEMPLE: BOUCLE D'ENTRAÎNEMENT (pseudo-code)
########################

def train_one_epoch(model, train_loader, optimizer, criterion, device="cpu"):
    model.train()

    for batch_idx, (imgs, masks, confs) in enumerate(train_loader):
        # imgs : (B, C, H, W)
        # masks: (B, H, W)
        # confs: (B, H, W)

        imgs  = imgs.to(device)
        masks = masks.to(device)
        confs = confs.to(device)

        # 1) Construire le target avec ignore_index basé sur la confidence
        targets_for_loss = []
        for b in range(imgs.shape[0]):
            conf_b = confs[b]   # (H, W)
            mask_b = masks[b]   # (H, W)

            ignore_mask = build_conf_ignore_mask(conf_b, threshold=2)
            target_mod  = apply_ignore_index_to_target(
                mask_b,
                ignore_mask,
                ignore_index=-100,
            )
            targets_for_loss.append(target_mod)

        targets_for_loss = torch.stack(targets_for_loss, dim=0)  # (B, H, W)

        # 2) Forward
        optimizer.zero_grad()
        logits = model(imgs)   # (B, num_classes, H, W) par exemple

        # 3) Loss (par ex. CrossEntropy2D avec ignore_index=-100)
        loss = criterion(logits, targets_for_loss)

        # 4) Backprop
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}, loss = {loss.item():.4f}")


########################
# 5. EXEMPLE: DATASET FLATTEN POUR RANDOM FOREST
########################

def build_rf_dataset(data_root):
    """
    Construit X, y pour RandomForest à partir de tous les patches
    (train + val, à adapter selon tes besoins).
    On utilise val_transform (pas d'augmentation aléatoire).
    """
    train_folder = os.path.join(data_root, "train")
    val_folder   = os.path.join(data_root, "val")

    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    train_dataset = MaridaDatasetLoader(
        folder=train_folder,
        bases=train_bases,
        transform=val_transform,
    )
    val_dataset = MaridaDatasetLoader(
        folder=val_folder,
        bases=val_bases,
        transform=val_transform,
    )

    rf_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset])
    rf_loader  = DataLoader(rf_dataset, batch_size=1, shuffle=False)

    all_X = []
    all_y = []

    for img, mask, conf in rf_loader:
        img  = img.squeeze(0)   # (C, H, W)
        mask = mask.squeeze(0)  # (H, W)
        conf = conf.squeeze(0)  # (H, W)

        X_rf, y_rf = flatten_for_rf(img, mask, conf, conf_threshold=2)
        all_X.append(X_rf)
        all_y.append(y_rf)

    X_all = torch.cat(all_X, dim=0)
    Y_all = torch.cat(all_y, dim=0)

    print("RF dataset : X =", X_all.shape, ", y =", Y_all.shape)
    return X_all, Y_all

In [79]:
# OVIA

def build_rf_dataset_from_splits(
    data_root,
    splits=("train", "val"),
    conf_threshold=2,
    max_patches=None,
):
    """
    Construit X, y pour RandomForest à partir des splits MARIDA
    en utilisant data/raw/patches + data/raw/splits.
    On réutilise exactement la même logique que make_dataloaders :
    - load_split_list
    - fix_base_name
    - do_img_conf_mask_exist
    """
    patches_root = os.path.join(data_root, "patches")
    splits_dir   = os.path.join(data_root, "splits")

    # 1) Lire les listes de patches pour chaque split
    all_bases = []
    for split in splits:
        split_file = os.path.join(splits_dir, f"{split}_X.txt")
        bases_raw = load_split_list(split_file)       # ex: "1-12-19_48MYU_0"
        bases_fix = [fix_base_name(b) for b in bases_raw]  # ex: "S2_.../S2_..._0"
        all_bases.extend(bases_fix)

    # 2) Garder seulement les patches où img + mask + conf existent
    all_bases = [
        b for b in all_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]
    print(f"{len(all_bases)} patches valides pour splits {splits}")

    if len(all_bases) == 0:
        raise RuntimeError(
            f"Aucun patch valide trouvé. Vérifie data_root={data_root} "
            "et le contenu de data/raw/patches + data/raw/splits."
        )

    # 3) Dataset MARIDA avec val_transform (pas d'augmentation aléatoire)
    rf_dataset = MaridaDatasetLoader(
        folder=patches_root,
        bases=all_bases,
        transform=val_transform,
    )

    rf_loader = DataLoader(
        rf_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )

    # 4) Flatten en pixels pour RF
    X_list, y_list = [], []

    for i, (img, mask, conf) in enumerate(rf_loader):
        img  = img.squeeze(0)   # (C, H, W)
        mask = mask.squeeze(0)  # (H, W)
        conf = conf.squeeze(0)  # (H, W)

        X_rf, y_rf = flatten_for_rf(img, mask, conf, conf_threshold)
        X_list.append(X_rf)
        y_list.append(y_rf)

        if max_patches is not None and (i + 1) >= max_patches:
            print(f"Limité à {max_patches} patches.")
            break

    X_all = torch.cat(X_list).cpu().numpy()
    Y_all = torch.cat(y_list).cpu().numpy()

    print("RF dataset : X =", X_all.shape, ", y =", Y_all.shape)
    return X_all, Y_all


In [80]:
data_root = os.path.join(os.getcwd(), "..", "data", "raw")
data_root = os.path.abspath(data_root)
train_loader, val_loader = make_dataloaders(data_root, batch_size=4)
test_loader = make_test_loader(data_root, batch_size=4)


Train : 694 patches valides.
Val   : 328 patches valides.


In [81]:
ds = train_loader.dataset

img1, mask1, conf1 = ds[0]
img2, mask2, conf2 = ds[0]

print("Same shape:", img1.shape, img2.shape)
print("Pixels exactly equal ?", torch.allclose(img1, img2))


Same shape: torch.Size([11, 256, 256]) torch.Size([11, 256, 256])
Pixels exactly equal ? False


In [ ]:
X_all, Y_all = build_rf_dataset_from_splits(
    data_root,
    splits=("train", "val"),  # ou ("train",) si tu veux garder "val" à part
    conf_threshold=2,
)

1022 patches valides pour splits ('train', 'val')
RF dataset : X = (66902349, 11) , y = (66902349,)


: 

In [ ]:
# Split sklearn en train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_all,
    Y_all,
    test_size=0.2,
    random_state=42,
    stratify=Y_all,
)

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    n_jobs=-1,
    random_state=0,
)

print("Entraînement du RandomForest...")
rf_model.fit(X_train, y_train)

y_train_pred = rf_model.predict(X_train)
y_val_pred   = rf_model.predict(X_val)

print("\n=== Performance RandomForest ===")
print(f"Accuracy train : {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Accuracy val   : {accuracy_score(y_val,   y_val_pred):.4f}\n")

print("Classification report (val) :")
print(classification_report(y_val, y_val_pred))

print("Matrice de confusion (val) :")
print(confusion_matrix(y_val, y_val_pred))
